# Brain Tumor Detection - Model Training

This notebook trains an EfficientNetB0 model to classify brain MRI images.

In [2]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models
import json
import numpy as np
import matplotlib.pyplot as plt
import os

## 1. Dataset Preparation
Ensure your dataset is in `data/train` and `data/val`.

In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

DATA_DIR = "../data" # Assuming notebook is in notebooks/
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

# Check if data exists
if not os.path.exists(TRAIN_DIR):
    print(f"Data directory not found at {TRAIN_DIR}. Please prepare your dataset.")
else:
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        TRAIN_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        VAL_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

Found 2562 files belonging to 4 classes.
Found 882 files belonging to 4 classes.


In [6]:
if os.path.exists(TRAIN_DIR):
    class_names = train_ds.class_names
    print(class_names)
    
    # Save label map
    label_map = {name: idx for idx, name in enumerate(class_names)}
    with open("../models/label_map.json", "w") as f:
        json.dump(label_map, f)
    print("Saved label_map.json")

['glioma', 'meningioma', 'no_tumor', 'pituitary']
Saved label_map.json


## 2. Augmentation and Normalization

In [7]:
normalization_layer = layers.Rescaling(1.0 / 255)

data_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1)
    ]
)

if os.path.exists(TRAIN_DIR):
    train_ds = train_ds.map(
        lambda x, y: (data_augmentation(normalization_layer(x)), y)
    )

    val_ds = val_ds.map(
        lambda x, y: (normalization_layer(x), y)
    )

## 3. Build Model

In [8]:
num_classes = 4 # Adjust based on dataset

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = normalization_layer(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,054,695 (15.47 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

## 4. Train (Phase 1)

In [9]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

if os.path.exists(TRAIN_DIR):
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        callbacks=[]
    )

Epoch 1/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 156s 2s/step - accuracy: 0.2818 - loss: 1.3595 - val_accuracy: 0.3390 - val_loss: 1.2979
Epoch 2/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.2963 - loss: 1.3602 - val_accuracy: 0.2086 - val_loss: 1.3296
Epoch 3/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 159s 2s/step - accuracy: 0.2931 - loss: 1.3583 - val_accuracy: 0.3583 - val_loss: 1.3005
Epoch 4/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 147s 2s/step - accuracy: 0.2963 - loss: 1.3532 - val_accuracy: 0.3390 - val_loss: 1.3139
Epoch 5/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 135s 2s/step - accuracy: 0.2982 - loss: 1.3598 - val_accuracy: 0.3390 - val_loss: 1.2927
Epoch 6/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.3052 - loss: 1.3598 - val_accuracy: 0.3390 - val_loss: 1.3347
Epoch 7/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 137s 2s/step - accuracy: 0.2857 - loss: 1.3671 - val_accuracy: 0.3390 - val_loss: 1.3076
Epoch 8/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.2951 - loss: 1.3605 - val_accuracy: 0.3390 - v

## 5. Fine Tuning

In [10]:
base_model.trainable = True

fine_tune_from = len(base_model.layers) // 2
for layer in base_model.layers[:fine_tune_from]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

if os.path.exists(TRAIN_DIR):
    history_fine = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        callbacks=[]
    )

Epoch 1/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 354s 4s/step - accuracy: 0.2900 - loss: 1.3893 - val_accuracy: 0.3390 - val_loss: 1.3110
Epoch 2/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 332s 4s/step - accuracy: 0.2998 - loss: 1.3729 - val_accuracy: 0.3390 - val_loss: 1.3147
Epoch 3/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 273s 3s/step - accuracy: 0.2990 - loss: 1.3693 - val_accuracy: 0.3390 - val_loss: 1.2951
Epoch 4/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 259s 3s/step - accuracy: 0.2978 - loss: 1.3593 - val_accuracy: 0.3390 - val_loss: 1.3088
Epoch 5/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 257s 3s/step - accuracy: 0.2935 - loss: 1.3598 - val_accuracy: 0.3583 - val_loss: 1.3347
Epoch 6/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 248s 3s/step - accuracy: 0.3091 - loss: 1.3558 - val_accuracy: 0.3583 - val_loss: 1.3294
Epoch 7/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 246s 3s/step - accuracy: 0.2787 - loss: 1.3553 - val_accuracy: 0.3583 - val_loss: 1.3179
Epoch 8/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 249s 3s/step - accuracy: 0.2881 - loss: 1.3467 - val_accuracy: 0.2086 - v

## 6. Save Model

In [11]:
model.save("../models/tumor_classifier.h5")
print("Model saved to ../models/tumor_classifier.h5")

Model saved to ../models/tumor_classifier.h5
